# Assemble datasets from simulations
Combine data from simulations of different network architectures

In [1]:
import numpy as np
import pandas as pd
import os
import pickle
from tqdm import tqdm
from joblib import Parallel, delayed
import re

from stoch_sim_model import *

In [2]:
# Set parameters
sim_kind = 'agent'
reg_model = ''
runs = '-5-'
comment = "acute_all-sparse-reg"

d = '/gscratch/scrubbed/oukogu/infoimmune/sim_output/no_cell_var/'

sim_sum_list = []
parameters_nets = []
prim_diff_bias_list = []
#sec_diff_bias_list = []
cell_series_list = []
# lineage_diff_nets = []

In [3]:
# Figure out which jobs didn't run:
d_rerun = '/gscratch/scrubbed/oukogu/infoimmune/sim_output/no_cell_var/raw/'
run_list = [int(re.search('sim_batch_(.*?)\.', f).group(1)) for f in os.listdir(d_rerun) if 'sim_batch' in f and comment in f and runs in f]
out = [str(x) for x in [k for k in np.arange(0, 1407)] if x not in run_list]
print(len(out))
print(' '.join((out)))

1
1342


In [4]:
num_cpu = 150
file_list = [f for f in os.listdir(os.path.join(d, "raw")) if runs in f and comment in f and 'sim_batch' in f]
num_files = len(file_list)

def import_dict_func(f,d):
    
    file_path = os.path.join(os.path.join(d, "raw"), f)
    with open(file_path, 'rb') as filename:  
        import_dict = pickle.load(filename)

    parameters = np.array(import_dict["parameters"])
    sim_sum = np.array(import_dict["summary_stats"])

    out = np.hstack((parameters, sim_sum))

    return out

# create dataframe of infection response statistics
var_names = np.concatenate((param_names_for_df, stat_names_for_df))
# mean_df = pd.DataFrame(np.vstack(Parallel(n_jobs = num_cpu, batch_size = max(int(num_files/num_cpu),1))(delayed(import_dict_func)(f = file_name, d = d) 
#                                                                                                         for file_name in file_list)), 
#                        columns = [i for i in var_names]).groupby(Na_reg + NE_reg + EM_reg + EE_reg + vir_vars, as_index=False).mean()
full_df = pd.DataFrame(np.vstack(Parallel(n_jobs = num_cpu, batch_size = max(int(num_files/num_cpu),1))(delayed(import_dict_func)(f = file_name, d = d) 
                                                                                                        for file_name in file_list)), 
                       columns = [i for i in var_names])

# # Save datasets
full_df.to_pickle(os.path.join(d, "raw", "stacked_full_data"+runs+"runs"+'-'+comment)+'.pkl')

In [5]:
# Create additional variables
virs = np.unique(full_df[['I_0','d_I','K_I','b_I','K_H','N_0']].values, axis = 0)

full_df['antigenicity_over_harm'] = antigenicity_over_harm(full_df)
full_df['T_pE_clear'] = full_df['T_pE_max'] - full_df['T_max_pI']
full_df['max_eM_fold'] = np.log10(1 + full_df['max_eM']/full_df['N_0'])
full_df['max_cM_fold'] = np.log10(1 + full_df['max_cM']/full_df['N_0'])
full_df['stim_pI'] = np.log10(1 + (full_df['p_load']/full_df['K_I'])/sim_duration)
full_df['stim_pHI'] = np.log10(1 + (full_df['int_pHI']/full_df['K_H']))
full_df['stim_pHE'] = np.log10(1 + (full_df['int_pHE']/full_df['K_H']))
full_df['max_pE_fold'] = np.log10(1 + full_df['max_pE']/full_df['N_0'])
full_df['log_min_pS'] = np.log10(full_df['min_pS']/full_df['S_0'])

# identify Biologically evidenced networks
keep_vars = ['harm_pI', 'harm_pS', 'max_eM_fold', 'max_cM_fold', 'T_eM_min',
             'T_min_pI', 'T_max_pI', 'T_pE_start', 'T_pE_clear',
             'stim_pI', 'stim_pHI', 'stim_pHE',
             'log_min_pS', 'antigenicity_over_harm']

In [6]:
# save data sets
full_infection_scenarios = []
mean_of_infection_scenarios = []
std_of_infection_scenarios = []
no_eff_data = [[] for i in np.arange(len(virs))]
b_S = d_S*S_0

for l, (I_0, d_I, K_I, b_I, K_H, N_0) in enumerate(tqdm(virs)):
    data = full_df.loc[(full_df["d_I"] == d_I)*(full_df["K_I"] == K_I)*(full_df["b_I"] == b_I)*(full_df["K_H"] == K_H)*(full_df["N_0"] == N_0)*(full_df["I_0"] == I_0), 
    ['b_I','d_I', 'K_I', 'I_0','S_0', 'N_0', 'd_S', 'K_H'] + Na_reg + NE_reg + EM_reg + EE_reg + keep_vars]

    # compute infection harm without T cell response
    no_eff_data[l] = lin_stoch_sim(N_0 = 0, I_0 = I_0, K_I = K_I, d_I = d_I, b_I = b_I,
                                   infection_model = "cancer" if b_I >= b_C else "acute")
    no_eff_stats = no_eff_data[l]["summary_stats"]

    data.loc[:,"harm_pI_noprotection"] = no_eff_stats[3]/(b_S*sim_duration)
    data.loc[:,"peff_clearance"] = (no_eff_stats[3] - data['harm_pI'].to_numpy())/(b_S*sim_duration)
    data.loc[:,"peff_toxicity"] = data['harm_pS'].to_numpy()/(b_S*sim_duration)

    mean_of_infection_scenarios.append(data.groupby(Na_reg + NE_reg + EM_reg + EE_reg + vir_vars, as_index=False).mean())
    std_of_infection_scenarios.append(data.groupby(Na_reg + NE_reg + EM_reg + EE_reg + vir_vars, as_index=False).std())
    full_infection_scenarios.append(data)

# stack datasets
pd.concat(mean_of_infection_scenarios).to_pickle('/gscratch/scrubbed/oukogu/infoimmune/sim_output/no_cell_var/summary_stats/mean/processed_data'+runs+'runs'+'-'+comment+'.pkl')
pd.concat(std_of_infection_scenarios).to_pickle('/gscratch/scrubbed/oukogu/infoimmune/sim_output/no_cell_var/summary_stats/std/processed_data'+runs+'runs'+'-'+comment+'.pkl')
pd.concat(full_infection_scenarios).to_pickle('/gscratch/scrubbed/oukogu/infoimmune/sim_output/no_cell_var/summary_stats/processed_full_data'+runs+'runs'+'-'+comment+'.pkl')

with open('/gscratch/scrubbed/oukogu/infoimmune/sim_output/no_cell_var/summary_stats/mean/list_processed_data'+runs+'runs'+'-'+comment+'.pkl', 'wb') as f:
    pickle.dump(mean_of_infection_scenarios, f)

with open('/gscratch/scrubbed/oukogu/infoimmune/sim_output/no_cell_var/summary_stats/std/list_processed_data'+runs+'runs'+'-'+comment+'.pkl', 'wb') as f:
    pickle.dump(std_of_infection_scenarios, f)

with open('/gscratch/scrubbed/oukogu/infoimmune/sim_output/no_cell_var/summary_stats/list_processed_full_data'+runs+'runs'+'-'+comment+'.pkl', 'wb') as f:
    pickle.dump(full_infection_scenarios, f)

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 147/147 [48:22<00:00, 19.75s/it]


In [7]:
# Clear memory
del full_df, mean_of_infection_scenarios, std_of_infection_scenarios, full_infection_scenarios